# 🏔️ ALTAY LLM — TEK TIKLA ÇALIŞTIR
**~2 saatte** Qwen 2.5 7B'yi Türkçe + İngilizce için özelleştirir.

## ▶️ YAPMAN GEREKEN
1. **Çalışma Zamanı → Tümünü çalıştır (Ctrl+F9)**
2. Token kutucuğuna token'ı yapıştır → Enter
3. ~2 saat bekle

> ⚠️ GPU uyarısı çıkarsa: **T4 GPU** seç → **Kaydet** → tekrar **Ctrl+F9**

In [ ]:
# ============================================================
# KURULUM
# ============================================================
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'YOK'}")

!pip install -q transformers datasets accelerate peft bitsandbytes trl huggingface_hub

from getpass import getpass
from huggingface_hub import login, HfApi, create_repo
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch
import os, gc, random

HF_TOKEN = getpass('Hugging Face token: ')
login(token=HF_TOKEN)
print("Hazir!")


In [ ]:
# ============================================================
# MODEL + TOKENIZER
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

total = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Model hazir! {total:.1f}M parametre egitilecek")


In [ ]:
# ============================================================
# VERI HAZIRLIGI
# ============================================================
print("Veri yukleniyor...")
oa = load_dataset("OpenAssistant/oasst1", split="train", streaming=True)

threads = {}
for item in oa:
    t = item["message_tree_id"]
    if t not in threads: threads[t] = []
    threads[t].append(item)
    if len(threads) >= 5000: break

SISTEM = "Sen, Altay LLM'sin. Turkce ve Ingilizce her konuda yardimci bir asistansin."

veri = []
for tid, msgs in threads.items():
    msgs.sort(key=lambda x: x["created_date"])
    dil = msgs[0].get("lang", "en")
    if dil not in ["tr", "en"]: continue
    if dil == "en" and len(veri) > 500: continue
    conv = [{"role": "system", "content": SISTEM}]
    for m in msgs:
        conv.append({"role": "assistant" if m["role"]=="assistant" else "user", "content": m["text"]})
    if len(conv) >= 2:
        veri.append(tokenizer.apply_chat_template(conv, tokenize=False))
    if len(veri) >= 2000: break

random.shuffle(veri)
n = int(len(veri)*0.95)

from datasets import Dataset
train_ds = Dataset.from_dict({"text": veri[:n]})
eval_ds = Dataset.from_dict({"text": veri[n:]})
print(f"{len(train_ds)} egitim, {len(eval_ds)} eval")


In [ ]:
# ============================================================
# EGITIM (~1.5-2 SAAT)
# ============================================================
print("EGITIM BASLIYOR! Colab'i kapatma.")

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=TrainingArguments(
        output_dir="./cikti",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=10,
        save_strategy="steps",
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        report_to="none",
        max_steps=1000,
    ),
    max_seq_length=2048,
    dataset_text_field="text",
    packing=True,
)

trainer.train()
print("EGITIM TAMAM!")


In [ ]:
# ============================================================
# KAYDET + YUKLE
# ============================================================
print("Model kaydediliyor...")
model.save_pretrained("./altay-son")
tokenizer.save_pretrained("./altay-son")

api = HfApi()
create_repo("cntzcn10/altay-llm", exist_ok=True)
api.upload_folder(repo_id="cntzcn10/altay-llm", folder_path="./altay-son", path_in_repo=".")
print("Model yayinda!")
print(f"https://huggingface.co/cntzcn10/altay-llm")


In [ ]:
# ============================================================
# TEST
# ============================================================
model.eval()
for soru in ["Merhaba, kimsin?", "Turkiye'nin baskenti neresi?"]:
    msg = [{"role": "system", "content": SISTEM}, {"role": "user", "content": soru}]
    girdi = tokenizer.apply_chat_template(msg, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        cikti = model.generate(girdi, max_new_tokens=100, temperature=0.7)
    cevap = tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)
    print(f"\nKullanici: {soru}")
    print(f"Altay: {cevap}")

print("\n🎉 ALTAY LLM HAZIR!")
